# Cek koneksi Colab ke DVWA lokal

Notebook ini hanya memeriksa koneksi, login, dan empat level keamanan DVWA melalui tunnel bertoken. Jalankan pada runtime Colab. Masukkan URL tunnel serta token secara privat; jangan simpan notebook beserta output atau token setelah pengujian. Target hanya instance DVWA milik sendiri.

In [ ]:
from getpass import getpass
from http.cookiejar import CookieJar
from urllib.error import HTTPError
from urllib.parse import urlencode, urlsplit
from urllib.request import HTTPCookieProcessor, Request, build_opener, urlopen
import re

BASE_URL = input('URL tunnel https://...trycloudflare.com: ').strip().rstrip('/')
LAB_TOKEN = getpass('LAB_TOKEN: ')
DVWA_PASSWORD = getpass('Password DVWA admin: ')
parts = urlsplit(BASE_URL)
if parts.scheme != 'https' or not parts.hostname or parts.path or parts.query:
    raise ValueError('Masukkan hanya URL HTTPS dasar dari tunnel.')
if not LAB_TOKEN or not DVWA_PASSWORD:
    raise ValueError('Token tunnel dan password DVWA harus diisi.')
print('Input siap; nilai rahasia tidak ditampilkan.')

In [ ]:
# Gateway harus menolak request yang tidak membawa token.
try:
    urlopen(BASE_URL + '/login.php', timeout=30)
except HTTPError as error:
    if error.code != 403:
        raise RuntimeError(f'Gateway memberi status {error.code}, bukan 403.')
    print('Tanpa token: 403 (sesuai)')
else:
    raise RuntimeError('Gateway menerima request tanpa token.')

# CookieJar menjaga sesi DVWA yang sama selama login dan pergantian level.
opener = build_opener(HTTPCookieProcessor(CookieJar()))
opener.addheaders = [('X-Lab-Token', LAB_TOKEN)]
with opener.open(BASE_URL + '/login.php', timeout=30) as response:
    login_page = response.read().decode('utf-8', 'replace')
    if response.status != 200 or 'DVWA' not in login_page:
        raise RuntimeError('Halaman login DVWA tidak terbuka melalui tunnel.')
print('Dengan token: 200, halaman login DVWA terbuka')

In [ ]:
def form_token(html):
    match = re.search(r"name=['\"]user_token['\"] value=['\"]([0-9a-f]+)", html)
    if not match:
        raise RuntimeError('Token form DVWA tidak ditemukan.')
    return match.group(1)

def post_form(path, values):
    data = urlencode(values).encode('utf-8')
    with opener.open(BASE_URL + path, data, timeout=30) as response:
        return response.read().decode('utf-8', 'replace')

home = post_form('/login.php', {
    'username': 'admin',
    'password': DVWA_PASSWORD,
    'Login': 'Login',
    'user_token': form_token(login_page),
})
if 'security.php' not in home:
    raise RuntimeError('Login DVWA gagal atau sesi tidak bertahan.')
print('Login DVWA melalui Colab: berhasil')

for level in ('low', 'medium', 'high', 'impossible'):
    with opener.open(BASE_URL + '/security.php', timeout=30) as response:
        security_page = response.read().decode('utf-8', 'replace')
    updated = post_form('/security.php', {
        'security': level,
        'seclev_submit': 'Submit',
        'user_token': form_token(security_page),
    })
    selected = re.search(r"<option value=\"([^\"]+)\" selected=['\"]selected['\"]>", updated)
    if not selected or selected.group(1) != level:
        raise RuntimeError(f'Level {level} tidak terkonfirmasi.')
    print(f'Level {level}: OK')

print('Milestone koneksi Colab ke empat level DVWA: TERUJI')

Jika semua sel berhasil, catat tanggal/jam dan empat status level pada log eksperimen, tanpa URL tunnel, token, password, atau cookie. Level terakhir pada sesi notebook adalah `Impossible`; pilih level yang sesuai sebelum pengujian berikutnya. Setelah selesai, hentikan tunnel sesuai `dvwa/README.md`.